# Notebook implementing DQN.

In [2]:
# The configurations here are tuned for the associated problem. To use this implementation on different systems,
# make necessary changes as directed.

In [1]:
using NBInclude
using FileIO

In [2]:
# Given the issue of versions of sub-dependencies of viscous streaming clashing with Flux.jl,
# an in-house deep neural network package was developed, named "Ankh.jl".

@nbinclude("Ankh.ipynb")

train_model (generic function with 1 method)

In [3]:
log_filename = "log_loss_diff"
open(string(log_filename,".txt"), "w") do file
end

In [4]:
mutable struct initialise{T<:Number, F<:Number}
    mem_size::T;
    discrete::Bool;
    mem_counter::T;
    state_memory::Array{F};
    newstate_memory::Array{F};
    action_memory::Array{F};
    reward_memory::Array{F};
    terminal_memory::Array{F};
end

In [5]:
mutable struct agent_initialise{T1<:Number, D1<: Number}
    action_space::Array{T1}
    gamma::D1
    epsilon::D1
    epsilon_decay::D1
    epsilon_end::D1
    alpha::D1
    batch_size::T1
end

In [6]:
# To use this for different systems, configure here.

const max_size = 1000000
const input_shape = 2
const n_actions = 16
const discrete = true

true

In [7]:
function initialisation()
    data = initialise(max_size, discrete, 0, zeros(max_size, input_shape), 
        zeros(max_size, input_shape), zeros(max_size, n_actions), zeros(max_size), zeros(max_size) )
    agentdata = agent_initialise([i for i in 1:n_actions], 0.99, 1.0, 0.99, 0.01, 0.001, 64)
    return data, agentdata
end    
data, agentdata = initialisation()

(initialise{Int64,Float64}(1000000, true, 0, [0.0 0.0; 0.0 0.0; … ; 0.0 0.0; 0.0 0.0], [0.0 0.0; 0.0 0.0; … ; 0.0 0.0; 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]), agent_initialise{Int64,Float64}([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], 0.99, 1.0, 0.99, 0.01, 0.001, 64))

In [8]:
function transitionstore(state, action, reward, state_n, done)
    ind = (data.mem_counter % data.mem_size)+1
    data.state_memory[ind,:] .= state
    data.newstate_memory[ind,:] .= state_n
    data.reward_memory[ind,:] .= reward
    data.terminal_memory[ind,:] .= 1 - Int8(done)
    actions = zeros(length(data.action_memory[1,:]))
    actions[action] = 1.0
    data.action_memory[ind,:] .= actions
    data.mem_counter += 1
end 

transitionstore (generic function with 1 method)

In [9]:
function sampling(batchsize)
    max_mem = min(data.mem_counter, data.mem_size)
    batch = rand(1:max_mem, batchsize)
    bstate = data.state_memory[batch,:]
    bstate_new = data.newstate_memory[batch,:]
    breward = data.reward_memory[batch,:]
    baction = data.action_memory[batch,:]
    bterminal = data.terminal_memory[batch,:]
    return bstate, baction, breward, bstate_new, bterminal
end

sampling (generic function with 1 method)

In [10]:
function remember(state, action, reward, new_state, done)
    transitionstore(state, action, reward, new_state, done)
end

remember (generic function with 1 method)

In [11]:
function action_choice(state)
    #println(state)
    if rand() < agentdata.epsilon
        action = rand(agentdata.action_space)
    else
        actions,_ = forward_propagation([state[1] state[2]], nn_param.param_learn, nn_param.activation_type)
        action = argmax(actions)[2] 
    end
    return action
end

action_choice (generic function with 1 method)

In [12]:
function learn()
        if data.mem_counter < agentdata.batch_size
            return
        end
        state, action, reward, new_state, done = sampling(agentdata.batch_size)                                                      
        action_values = reshape(agentdata.action_space,(n_actions,1))
        action_indices = sum(action .* action_values', dims=2)  
    
        q_eval,_ = forward_propagation(state, nn_param.param_learn, nn_param.activation_type)
        q_next,_ = forward_propagation(new_state, nn_param.param_learn, nn_param.activation_type)
        
        q_target = deepcopy(q_eval)
        
        batch_index = reshape([i for i in 1:agentdata.batch_size],(agentdata.batch_size,1))
        
        CartIndx = [CartesianIndex(batch_index[i,1],Int64.(action_indices[i,1])) for i in 1:length(batch_index)]
        q_target[CartIndx] = (reward + agentdata.gamma*(maximum(q_next, dims = 2).*done))  

        # Some neural network parameters can be toggled with here.
    
        nn_param.param_learn = train_model(nn_param.param_learn, state, q_target, nn_param.layers, 
                                nn_param.activation_type, nn_param.loss_fcn;lr=0.0001,epochs=15,
                                verbose=false, shuffle_data = true, steps_per_epoch = 2,
                                batch_size= nothing, optimiser = "Adam")
                
        if agentdata.epsilon > agentdata.epsilon_end
           agentdata.epsilon = agentdata.epsilon*agentdata.epsilon_decay
        
        else agentdata.epsilon_end 
        end
end

learn (generic function with 1 method)

In [13]:
function savemodel(arg)
    save(string(arg,".jld2"), "data", nn_param.param_learn)
end

savemodel (generic function with 1 method)

In [14]:
function loadmodel(arg)
    return load(string(arg,".jld2"))["data"];
end

loadmodel (generic function with 1 method)

In [15]:
mutable struct parameters_learn{T<:Dict{Any, Any}, B<:Vector{String}, D<:Vector{Int64},P<:String}
    param_learn::T
    activation_type::B
    layers::D
    loss_fcn::P
end

In [16]:
# Change the neural network parameters here.

nn_param = parameters_learn(initialise_parameters([input_shape, 128, 128, 128, n_actions], "he_uniform"),["ReLu", "ReLu", "ReLu", "Linear"],[input_shape, 128, 128, 128, n_actions],"mse")

parameters_learn{Dict{Any,Any},Array{String,1},Array{Int64,1},String}(Dict{Any,Any}("W2" => [0.0634052329849702 0.04272915697307311 … -0.19424897347938622 0.12026684525248366; 0.14695259081434403 0.14505429913633738 … 0.05426966293189256 -0.18713294137378889; … ; 0.006864111979637211 0.012309809316653875 … 0.09502456822502708 -0.03009540934327387; 0.07188894339482305 0.19903500901360333 … -0.1502037942588012 -0.033536393202286074],"W3" => [0.20005218498617072 -0.17643735023098084 … -0.1971524491449612 0.11261557672392097; 0.12420389924662723 -0.06583250139024549 … 0.09372012969548466 0.05840005130967413; … ; -0.08025048588110131 0.022308767801587887 … 0.11714426695170416 -0.011685016219627253; 0.10178219000640726 0.03930910999864984 … -0.21402518913808832 0.12543595582708117],"W4" => [-0.14934163617841922 -0.011454323993370513 … 0.03959882624282557 0.0923463449447452; -0.010778211530374004 0.20864802216604367 … -0.13270087493691426 -0.04607925542815505; … ; -0.08005989850978998 0.07720

In [17]:
nn_param.param_learn

Dict{Any,Any} with 8 entries:
  "W2" => [0.0634052 0.0427292 … -0.194249 0.120267; 0.146953 0.145054 … 0.0542…
  "W3" => [0.200052 -0.176437 … -0.197152 0.112616; 0.124204 -0.0658325 … 0.093…
  "W4" => [-0.149342 -0.0114543 … 0.0395988 0.0923463; -0.0107782 0.208648 … -0…
  "b3" => [0.0; 0.0; … ; 0.0; 0.0]
  "b4" => [0.0; 0.0; … ; 0.0; 0.0]
  "W1" => [0.125411 0.155017; -1.70269 -0.593313; … ; -1.47879 0.309527; 0.9455…
  "b2" => [0.0; 0.0; … ; 0.0; 0.0]
  "b1" => [0.0; 0.0; … ; 0.0; 0.0]